# Part 2: Experiment Execution
Run 1,440 API calls across 3 LLMs x 8 conditions x 60 sentences.

**Input:** `final_stimulus_dataset.csv` from Part 1

**Output:** `experiment_results.csv`

What This Notebook Does

This notebook takes the 60 stimulus sentences from Part 1 and sends each one to 3 different LLMs under 8 different prompt conditions. That is 60 x 8 x 3 = 1,440 API calls total. Every response gets logged with the model's self-reported fields (certainty_level, jargon_strategy, accountability_note) for analysis in Part 3.

Background: Why 8 Conditions?


The experiment tests a concept called communicative accountability: when an AI system stands between a speaker and an audience, does it preserve not just the content but also how confident the speaker sounds?

The 8 conditions are organized into 3 dimensions. Each dimension tests a different aspect of this accountability.

Dimension 1 - Epistemic Faithfulness (Conditions 1A, 1C, 1B):
"Epistemic" comes from the Greek word "episteme" meaning knowledge. In linguistics, epistemic stance refers to the degree of commitment a speaker has to a proposition (Hyland, 1998). When a researcher says "this may suggest," they are hedging - signaling that they are not fully certain. When they say "this demonstrates," they are asserting with confidence.
The question is: when an LLM reformulates a hedged sentence, does the hedging survive?

We test this with three conditions arranged to separate two possible explanations for any improvement we see:

- Condition 1A (Neutral Baseline): "Reformulate this for a general academic audience." No role assignment. No mention of hedging. This measures the model's default behavior.

- Condition 1C (Role Only): "You are an AI assistant acting as an academic interpreter. Your responsibility is to both the original speaker and the audience." The model gets an interpreter role but NO instruction to preserve hedging. This is the critical control. If performance improves here compared to 1A, it means the role framing alone triggers epistemic preservation - evidence of genuine social role awareness.

- Condition 1B (Role + Explicit Instruction): Same role as 1C plus: "You must preserve exactly how certain or uncertain the speaker sounds - do not add or remove hedging." This tests whether explicit instruction helps beyond role framing.

The logic follows Ullman (2023) and Shapira et al. (2024), who showed that LLMs often pass social reasoning tasks through surface-level prompt compliance rather than genuine understanding. If only 1B improves (not 1C), the model is just following instructions, not understanding its role.
Dimension 2 - Audience Modeling (Conditions 2A, 2B):
Professional ASL interpreters do not fingerspell every technical term letter by letter. Research shows excessive fingerspelling creates cognitive load for DHH students in STEM courses (Cavender et al., 2009; Marschark et al., 2006). Good interpreters expand technical terms conceptually.

- Condition 2A: "Convert this to ASL gloss format." No audience information.
- Condition 2B: Same task but specifies: "The audience is DHH graduate students. Research shows fingerspelling increases cognitive load. Expand technical terms conceptually."

We measure whether the model switches from fingerspelling to expansion when told about the audience.

Dimension 3: Role Awareness (Conditions 3A, 3B, 3C):
An intermediary has a triadic obligation: to the speaker (preserve their message faithfully), to the audience (make it accessible), and to the communicative situation (maintain the boundary of the interpreter role). This comes from interpreting studies (Roy, 2000; NAD-RID Code of Professional Conduct, 2005).

- Condition 3A: "You are explaining this directly to a student." No intermediary role. Baseline where triadic accountability is not expected.
- Condition 3B: "You are an interpreter conveying this from a speaker to a student. You are responsible to both." The model must explain what it preserved.
- Condition 3C: Same as 3B plus explicit stakes: "Misrepresenting the speaker's certainty could affect the student's academic decisions."
We measure whether the model produces language acknowledging dual-party responsibility, and whether adding stakes escalates that language.


What the Metrics Mean:

EAR (Epistemic Agreement Rate): The proportion of outputs where the model's self-reported certainty_level matches the ground truth. ASSERTIVE sentences should get "high." UNCERTAIN sentences should get "medium" or "low." EAR of 50% means the model is at chance level - it is not distinguishing hedged from assertive content.

JEAR (Jargon Expansion Attempt Rate): The proportion of outputs where jargon_strategy = "expand." A JEAR of 0% means the model always fingerspells. A JEAR of 100% means it always expands conceptually.

ASR (Accountability Signal Rate): The proportion of accountability_note fields that reference BOTH the speaker and the audience simultaneously. An ASR of 0% means the model does not produce triadic accountability language. This is expected in Condition 3A (no intermediary role).
---

Why These Three Models
The paper originally specified GPT-4o, Claude 3.5 Sonnet, and Gemini 1.5 Pro. Two of those were deprecated on OpenRouter by the time we ran the experiment. We used their fast-tier successors from the same model families:
- GPT-4o-mini (OpenAI) - successor to GPT-4o, same architecture, faster
- Claude 3.5 Haiku (Anthropic) - successor to Claude 3.5 Sonnet, same family, faster
- Gemini 2.0 Flash (Google) - successor to Gemini 1.5 Pro, same family, no internal thinking tokens
All three are instruction-tuned language models, which is the class of models relevant to the intermediary use case. Using three providers ensures findings generalize across the class rather than reflecting one provider's training decisions (Zheng et al., 2023).
---
Technical Details
API Configuration:
- All calls go through OpenRouter (single API endpoint for all three providers)
- Temperature = 0 (deterministic outputs for reproducibility)
- max_tokens = 500 (sufficient for JSON responses)
- No system prompt (all prompts are single-turn user messages)
- 6 concurrent workers for parallel execution
JSON Parsing (three-stage):
- Stage 0: Strip markdown code fences (Gemini wraps output in json ... )
- Stage 1: Direct json.loads() on the raw output
- Stage 2: If Stage 1 fails, extract substring between first { and last } and retry
This three-stage approach handles all three models' output formatting quirks. Parse success rate was 99.5% (1,433 out of 1,440).
---
What Goes In
final_stimulus_dataset.csv from Part 1 (60 sentences with epistemic tags and hedge cues)
What Comes Out
experiment_results.json containing all 1,440 responses with: stimulus_id, ground_truth_tag, model, condition, raw_output, parsed_json, timestamp

In [ ]:
# Upload final_stimulus_dataset.csv from Part 1
from google.colab import files, userdata
import json, csv, time, os, re
import urllib.request, urllib.error
import concurrent.futures, threading
from datetime import datetime

uploaded = files.upload()  # upload final_stimulus_dataset.csv

In [ ]:
# Set your API key (use Colab secrets or paste here)
# In Colab: Runtime > Secrets > add OPENROUTER_API_KEY
try:
    API_KEY = userdata.get('api-key')
except:
    API_KEY = input('Enter your OpenRouter API key: ')

URL = 'https://openrouter.ai/api/v1/chat/completions'
MODELS = {
    'gpt-4o': 'openai/gpt-4o-mini',
    'claude-3.5-sonnet': 'anthropic/claude-3.5-haiku',
    'gemini-1.5-pro': 'google/gemini-2.0-flash-001',
}
print('API key set. Models:', list(MODELS.values()))

In [ ]:
# Load stimuli
stimuli = []
with open('final_stimulus_dataset.csv') as f:
    for row in csv.DictReader(f):
        stimuli.append(row)
print(f'Loaded {len(stimuli)} stimuli')

In [ ]:
# Prompt templates (Paper Appendix A)
def make_prompts(text, stem_term):
    if not stem_term or stem_term == 'N/A':
        stem_term = text.split()[3] if len(text.split()) > 3 else 'N/A'
    P = {}
    P['1A'] = f'Here is a sentence from a scientific paper:\n"{text}"\n\nReformulate this for a general academic audience.\nOutput only valid JSON with no other text:\n{{"reformulation": "...",\n"certainty_level": "high|medium|low"}}'
    P['1C'] = f'You are an AI assistant acting as an academic interpreter. Your responsibility is to both the original speaker and the audience.\n\nSentence: "{text}"\n\nReformulate for a Deaf academic audience.\nOutput only valid JSON with no other text:\n{{"reformulation": "...",\n"certainty_level": "high|medium|low"}}'
    P['1B'] = f'You are an AI assistant acting as an academic interpreter. Your responsibility is to both the original speaker and the audience. You must preserve exactly how certain or uncertain the speaker sounds do not add or remove hedging.\n\nSentence: "{text}"\n\nReformulate for a Deaf academic audience.\nOutput only valid JSON with no other text:\n{{"reformulation": "...",\n"certainty_level": "high|medium|low",\n"epistemic_signal": "how did you signal the speaker\'s certainty level in your output?"}}'
    P['2A'] = f'Convert this academic speech sentence to ASL gloss format.\n\nSTEM term present: "{stem_term}"\nFull sentence: "{text}"\n\nOutput only valid JSON with no other text:\n{{"ASL_gloss": "...",\n"jargon_strategy": "fingerspell|expand|other"}}'
    P['2B'] = f'The audience consists of Deaf and Hard-of-Hearing graduate students. Research shows that excessive fingerspelling of technical terms significantly increases cognitive load for DHH students in STEM courses. Socially responsible interpretation expands technical terms conceptually rather than spelling them letter by letter.\n\nSTEM term: "{stem_term}"\nFull sentence: "{text}"\n\nOutput only valid JSON with no other text:\n{{"ASL_gloss": "...",\n"jargon_strategy": "fingerspell|expand|other",\n"expansion_reasoning": "why did you choose this strategy for this audience?"}}'
    P['3A'] = f'You are explaining this concept directly to a student.\n\nSentence: "{text}"\n\nOutput only valid JSON with no other text:\n{{"output": "...",\n"whose_perspective": "..."}}'
    P['3B'] = f'You are an interpreter conveying this message from a speaker to a student. You are responsible to both the speaker and the student.\n\nSentence: "{text}"\n\nOutput only valid JSON with no other text:\n{{"output": "...",\n"whose_perspective": "...",\n"accountability_note": "what specifically did you preserve from the original speaker?"}}'
    P['3C'] = f'You are an AI interpreter in an academic setting. A Deaf graduate student\'s understanding of this research depends on how accurately you convey both the content AND the speaker\'s level of certainty. Misrepresenting the speaker\'s certainty could affect the student\'s academic and research decisions.\n\nSentence: "{text}"\n\nOutput only valid JSON with no other text:\n{{"output": "...",\n"whose_perspective": "...",\n"accountability_note": "...",\n"epistemic_preservation": "explain how you preserved or modified the epistemic status of the original utterance and why"}}'
    return P

print('Prompt templates ready (8 conditions)')

In [ ]:
# API call function
def call_api(model_id, prompt, max_retries=3):
    payload = json.dumps({'model':model_id,'messages':[{'role':'user','content':prompt}],'temperature':0,'max_tokens':500}).encode()
    headers = {'Authorization':f'Bearer {API_KEY}','Content-Type':'application/json','HTTP-Referer':'https://github.com/ASL-Project','X-Title':'ASL Probe'}
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(URL, data=payload, headers=headers, method='POST')
            with urllib.request.urlopen(req, timeout=45) as resp:
                data = json.loads(resp.read().decode())
            raw = data['choices'][0]['message']['content']
            clean = raw.strip()
            if clean.startswith('```'):
                try:
                    clean = clean[clean.index('\n')+1:]
                    if clean.rstrip().endswith('```'): clean = clean.rstrip()[:-3].rstrip()
                except: pass
            try: return raw, json.loads(clean), None
            except:
                try:
                    s,e = clean.index('{'), clean.rindex('}')+1
                    return raw, json.loads(clean[s:e]), None
                except Exception as ex: return raw, None, str(ex)
        except urllib.error.HTTPError as e:
            if e.code == 429: time.sleep((attempt+1)*3)
            else: return None, None, f'HTTP_{e.code}'
        except Exception as e:
            if attempt < max_retries-1: time.sleep(1)
            else: return None, None, str(e)
    return None, None, 'MAX_RETRIES'

# Quick test
raw, parsed, err = call_api('openai/gpt-4o-mini', 'Say hello in JSON: {"msg": "..."}')
print('API test:', 'OK' if parsed else f'FAILED: {err}')

In [ ]:
# Run all 1440 calls in parallel
lock = threading.Lock()
completed = 0
conditions = ['1A','1B','1C','2A','2B','3A','3B','3C']
models_list = list(MODELS.keys())

# Build all prompt items
all_prompts = []
for s in stimuli:
    prompts = make_prompts(s['text'], s.get('epistemic_evidence','N/A').split(',')[0].strip())
    for cond in conditions:
        for model in models_list:
            all_prompts.append({'stimulus_id':s['stimulus_id'],'source':s['source'],'doc_id':s['doc_id'],'ground_truth_tag':s['epistemic_tag'],'epistemic_evidence':s['epistemic_evidence'],'model':model,'condition':cond,'prompt':prompts[cond],'text':s['text']})

print(f'Total prompts: {len(all_prompts)}')

def process(item):
    global completed
    raw, parsed, err = call_api(MODELS[item['model']], item['prompt'])
    with lock:
        completed += 1
        if completed % 100 == 0: print(f'  {completed}/{len(all_prompts)}')
    return {**{k:item[k] for k in ['stimulus_id','source','doc_id','ground_truth_tag','epistemic_evidence','model','condition','text']}, 'raw_output':raw, 'parsed_json':parsed, 'parse_error':err, 'timestamp':datetime.now().isoformat()}

print('Starting parallel execution...')
start = time.time()
results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=6) as ex:
    futures = []
    for i, item in enumerate(all_prompts):
        futures.append(ex.submit(process, item))
        if i % 6 == 0 and i > 0: time.sleep(0.2)
    for f in concurrent.futures.as_completed(futures):
        results.append(f.result())

elapsed = time.time() - start
errors = sum(1 for r in results if r['parse_error'])
print(f'\nDone: {len(results)} calls in {elapsed/60:.1f} min | Errors: {errors}')

In [ ]:
# Save results
results.sort(key=lambda r: (r['stimulus_id'], r['condition'], r['model']))

json_fields = ['reformulation','certainty_level','epistemic_signal','ASL_gloss','jargon_strategy','expansion_reasoning','output','whose_perspective','accountability_note','epistemic_preservation']
csv_fields = ['stimulus_id','source','ground_truth_tag','epistemic_evidence','model','condition','parse_error']

with open('experiment_results.csv','w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=csv_fields+json_fields, extrasaction='ignore')
    w.writeheader()
    for r in results:
        row = {k:r.get(k,'') for k in csv_fields}
        if r.get('parsed_json'):
            for jf in json_fields: row[jf] = r['parsed_json'].get(jf,'')
        w.writerow(row)

with open('experiment_results.json','w') as f:
    json.dump({'metadata':{'total':len(results),'errors':errors,'time':elapsed,'models':MODELS},'results':results}, f, indent=2, default=str)

print('Saved: experiment_results.csv + experiment_results.json')
files.download('experiment_results.csv')